In [ ]:
pip install requests pandas xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd

# Lista krajów do analizy
countries = {
    'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria', 'HRV': 'Croatia',
    'CYP': 'Cyprus', 'CZE': 'Czech Republic', 'DNK': 'Denmark', 'EST': 'Estonia',
    'FIN': 'Finland', 'FRA': 'France', 'DEU': 'Germany', 'GRC': 'Greece',
    'HUN': 'Hungary', 'IRL': 'Ireland', 'ITA': 'Italy', 'LVA': 'Latvia',
    'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'MLT': 'Malta', 'NLD': 'Netherlands',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania', 'SVK': 'Slovakia',
    'SVN': 'Slovenia', 'ESP': 'Spain', 'SWE': 'Sweden'
}

# Wskaźniki do pobrania
indicators = {
    'NE.IMP.GNFS.CD': 'Import (USD)',
    'NE.EXP.GNFS.CD': 'Export (USD)'
}

start_year = 2004
end_year = 2024

# Funkcja do pobierania danych z API Banku Światowego
def fetch_data(country_code, indicator_code):
    url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}?date={start_year}:{end_year}&format=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if len(data) > 1 and isinstance(data[1], list):
            return data[1]
    return []

# Inicjalizacja słownika do przechowywania danych
data_dict = {indicator: [] for indicator in indicators}

# Pobieranie danych dla każdego kraju i wskaźnika
for country_code, country_name in countries.items():
    for indicator_code, indicator_name in indicators.items():
        data = fetch_data(country_code, indicator_code)
        for entry in data:
            year = entry.get('date')
            value = entry.get('value')
            if year and value is not None:
                data_dict[indicator_code].append({
                    'Year': int(year),
                    'Country': country_name,
                    indicator_name: value
                })

# Pobieranie danych globalnych (World)
global_data = {indicator: {} for indicator in indicators}
for indicator_code in indicators.keys():
    data = fetch_data('WLD', indicator_code)
    for entry in data:
        year = entry.get('date')
        value = entry.get('value')
        if year and value is not None:
            global_data[indicator_code][int(year)] = value

# Tworzenie DataFrame dla każdego wskaźnika
dfs = {}
for indicator_code, records in data_dict.items():
    df = pd.DataFrame(records)
    if not df.empty:
        df_pivot = df.pivot(index='Year', columns='Country', values=indicators[indicator_code])
        dfs[indicator_code] = df_pivot

# Obliczanie udziału procentowego w imporcie i eksporcie światowym
share_dfs = {}
for indicator_code, df_pivot in dfs.items():
    share_df = df_pivot.copy()
    for year in share_df.index:
        if year in global_data[indicator_code]:
            global_value = global_data[indicator_code][year]
            share_df.loc[year] = (share_df.loc[year] / global_value) * 100
    share_dfs[indicator_code] = share_df

# Tworzenie arkusza informacyjnego
info_data = {
    "Description": [
        "Source: World Bank Open Data",
        "Indicators:",
        "- Import of Goods and Services (USD)",
        "- Export of Goods and Services (USD)",
        "Years Covered: {}-{}".format(start_year, end_year),
        "Countries Covered: " + ", ".join(countries.values())
    ]
}
df_info = pd.DataFrame(info_data)

# Zapis do pliku Excel z odpowiednimi arkuszami
output_file = "world_bank_trade_data.xlsx"
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    df_info.to_excel(writer, sheet_name="0 - Info", index=False)
    for indicator_code, df_pivot in dfs.items():
        sheet_name = indicators[indicator_code]
        df_pivot.to_excel(writer, sheet_name=sheet_name)
        share_sheet_name = sheet_name.replace(" (USD)", "") + " Share"
        share_dfs[indicator_code].to_excel(writer, sheet_name=share_sheet_name)

print(f"Plik zapisano jako: {output_file}")


Plik zapisano jako: world_bank_trade_data_corrected.xlsx
